# Booklet 4: Corpus Building

Continuing with the dependency parsing shown in the previous notebook, we will now show the `corpus.py` module, which is what will let us build a treebank (a corpus of CoNLL-U formatted text). In terms of our high level pipeline, along with the `dependency.py` module, this module can be seen as the implementation of the final CG3 (Dependency Parsing) -> Treebank (CoNLL-U) step in our pipeline. 

To build our corpus, we will use a small subset of 50 OPD sentences located at [`data/parallel/`](../../data/parallel/), where every sentence has at least 1 ambiguity, and has every token parsed by the FST. This was a subset preparsed for for expository and developmental purposes, while if we were to build a corpus from purely raw data, we might not expect every token to get a FST parse. 

In [ ]:
# we will setup all the imports and paths in this block
from __future__ import annotations
import os, sys, io
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).resolve().parents[1] # ../../
SRC_DIR = REPO_ROOT / "src"
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from src.corpus import (
    cg3_to_conllu_batch,
    append_sentence,
    validate_ud,
    visualise_conllu,
    delete_sentence,
)
from src.dependency import parse_dependencies
from src.disambiguation import load_fst_parser, fst_parse_word

# paths to the english and ojibwe sentences
OJ_PATH = (REPO_ROOT / "data" / "parallel" / "oj_50.txt").resolve()
EN_PATH = (REPO_ROOT / "data" / "parallel" / "en_50.txt").resolve()

# fst + grammar paths
FST_PATH = (REPO_ROOT / "data" / "fst" / "ojibwe7.fomabin").resolve()
DISAMBIG_CG_PATH = (REPO_ROOT / "data" / "rules" / "disambiguation.cg3").resolve()
DEPENDENCY_CG_PATH = (REPO_ROOT / "data" / "rules" / "dependency.cg3").resolve()

# Output corpus file (feel free to change the name or location)
CORPUS_PATH = (REPO_ROOT / "data" / "treebanks" / "oj_50_treebank.conllu").resolve()
CORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)

# Optional UD validator (edit or leave default)
UD_VALIDATOR = (REPO_ROOT / "ud-tools" / "validate.py").resolve()


In [5]:
# make sure paths exist
for p in [OJ_PATH, EN_PATH, FST_PATH, DISAMBIG_CG_PATH, DEPENDENCY_CG_PATH]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing required file: {p}")
    
# load fst
fst = load_fst_parser(str(FST_PATH))


FST file is /Users/matthias/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/ojibwe7.fomabin


### Building the corpus

Now that we set up our paths, we can start to build the corpus by going through every line, parsing dependencies using `parse_dependencies()`, then converting it to CoNLL-U and appending it to the corpus in one go using `cg3_to_conllu_batch()`. 

In [6]:
# build the corpus from 50 sentences
RESET_CORPUS = True  # set true since we are building a new file

if RESET_CORPUS and CORPUS_PATH.exists():
    CORPUS_PATH.unlink()
    print("Reset:", CORPUS_PATH.name)

oj_lines = Path(OJ_PATH).read_text(encoding="utf-8").splitlines()
en_lines = Path(EN_PATH).read_text(encoding="utf-8").splitlines()
assert len(oj_lines) == len(en_lines), "OJ/EN line count mismatch"

kept, failures = 0, 0
first_errors = []

for i, (oj, en) in enumerate(zip(oj_lines, en_lines), start=1):
    try:
        # parse the deps
        dep_cg3 = parse_dependencies(
            sentence=oj,
            dependency_grammar=str(DEPENDENCY_CG_PATH),
            disambiguation_grammar=str(DISAMBIG_CG_PATH),
            fst=fst,
            verbose=False
        )
        # convert and append to corpus (auto sent_id)
        cg3_to_conllu_batch(dep_cg3, corpus_path=str(CORPUS_PATH))
        kept += 1
    except Exception as e:
        failures += 1
        if len(first_errors) < 5:
            first_errors.append((i, oj, str(e)))

print(f"Done. Added {kept} sentences. Failures: {failures}")
if first_errors:
    print("\nSample errors:")
    for idx, s, err in first_errors:
        print(f"- line {idx}: {s}\n  {err}\n")


✓ appended sentence #1 to oj_50_treebank.conllu
✓ appended sentence #2 to oj_50_treebank.conllu
✓ appended sentence #3 to oj_50_treebank.conllu
✓ appended sentence #4 to oj_50_treebank.conllu
✓ appended sentence #5 to oj_50_treebank.conllu
✓ appended sentence #6 to oj_50_treebank.conllu
✓ appended sentence #7 to oj_50_treebank.conllu
✓ appended sentence #8 to oj_50_treebank.conllu
✓ appended sentence #9 to oj_50_treebank.conllu
✓ appended sentence #10 to oj_50_treebank.conllu
✓ appended sentence #11 to oj_50_treebank.conllu
✓ appended sentence #12 to oj_50_treebank.conllu
✓ appended sentence #13 to oj_50_treebank.conllu
✓ appended sentence #14 to oj_50_treebank.conllu
✓ appended sentence #15 to oj_50_treebank.conllu
✓ appended sentence #16 to oj_50_treebank.conllu
✓ appended sentence #17 to oj_50_treebank.conllu
✓ appended sentence #18 to oj_50_treebank.conllu
✓ appended sentence #19 to oj_50_treebank.conllu
✓ appended sentence #20 to oj_50_treebank.conllu
✓ appended sentence #21 to oj

Now let's take a quick look at the corpus that we just built, to see the general structure. 

As we can see below, each sentence has its own sentence id, the corresponding Ojibwe text, and the CoNLL-U formatted output. 

In [4]:
# show the first 30 lines of the corpus file
snippet = Path(CORPUS_PATH).read_text(encoding="utf-8").splitlines()[:30]
print("\n".join(snippet))


FileNotFoundError: [Errno 2] No such file or directory: '/Users/matthias/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/treebanks/oj_50_treebank.conllu'

### Visualizing and deleting sentences

Let's now hit two birds with one stone, by first visualizing the first sentence, deleting it, and then visualizing the new first sentence in the corpus.

As we will be able to see below, the `delete_sentence()` function automatically renumbers the corpus, so that the corpus will always start from `sent_id = 1` and grow in ascending order.

In terms of the visualization, the output may slightly differ depending on when you run this notebook, since development is not fully complete on the dependency parsing grammar yet. In general, we see that verbs are linked to subjects / objects, while nouns are linked to determiners / relative clauses. For full information on the dependencies, head to [03_grammar_modules.mb](../03_grammar_modules.md). 

In [ ]:
# visualise sentence 1
visualise_conllu(str(CORPUS_PATH), sent_no=1, compact=True, collapse_punct=True)

# delete sentence #1
deleted = delete_sentence(str(CORPUS_PATH), sent_id=1)

# New sentence 1 is old sentence 2
visualise_conllu(str(CORPUS_PATH), sent_no=1, compact=True, collapse_punct=True)


visualising sentence 1 from oj_50_treebank.conllu

✓ removed sent_id 1 and renumbered the file

visualising sentence 1 from oj_50_treebank.conllu

### Validating the corpus

Finally, we will run the UD validator on our corpus. This validator comes from [the UD tools repo](https://github.com/universaldependencies/tools), a public repository maintained by the Universal Dependencies project which contains many useful tools for working with treebanks. Since Ojibwe does not have its own designated UD language, we can use the general `lang="ud"`, which will nonetheless give us concrete formatting feedback. Although, part of building a complete UD treebank for Ojibwe would include making a concrete Ojibwe UD language, so later this might turn into `lang=oj`. 

Again, depending on when the following cell is run, the output might be different. Currently there are some formatting errors due to imperfect CoNLL-U conversion, and others that are due to incomplete development of the dependency grammar. Running the validator is an important step in pushing treebank development, and the final goal will be to have no formatting errors without having to manually correct anything.

In [ ]:
# Validate with UD tools
if UD_VALIDATOR.exists():
    validate_ud(str(CORPUS_PATH), lang="ud", validator=str(UD_VALIDATOR))
else:
    print("UD validator not found at:", UD_VALIDATOR)

⚠️  validator error(s) in oj_50_treebank.conllu

[Line 23 Sent 3 Node 3]: [L4 Syntax unknown-deprel] Unknown DEPREL label: 'acl:relcl'

The following 37 relations are currently permitted in language :
acl, advcl, advmod, amod, appos, aux, case, cc, ccomp, clf, compound, conj, cop, csubj, dep, det, discourse, 
dislocated, expl, fixed, flat, goeswith, iobj, list, mark, nmod, nsubj, nummod, obj, obl, orphan, parataxis, punct,
reparandum, root, vocative, xcomp
If a language needs a relation subtype that is not documented in the universal guidelines, the relation
must have a language-specific documentation page in a prescribed format.
See https://universaldependencies.org/contributing_language_specific.html for further guidelines.
Documented dependency relations can be specifically turned on/off for each language in which they are used.
See https://quest.ms.mff.cuni.cz/udvalidator/cgi-bin/unidep/langspec/specify_deprel.pl for details.


[Line 41 Sent 5 Node 5]: [L4 Syntax unknown-deprel] Unknown DEPREL label: 'acl:relcl'
[Line 41 Sent 5 Node 6]: [L3 Syntax leaf-det] 'det' not expected to have children (6:'aw:det --> 
5:gaa-pabaamanokiid:acl)
[Line 62 Sent 7 Node 7]: [L3 Syntax punct-causes-nonproj] Punctuation must not cause non-projectivity of nodes 
[Node<7#3, 'iw>, Node<7#6, niningwan>]
[Line 94 Sent 11 Node 1]: [L4 Syntax unknown-deprel] Unknown DEPREL label: 'neg'
[Line 163 Sent 20 Node 1]: [L3 Syntax too-many-objects] Multiple direct objects [2, 4] ('ingo-minikwaajigan', 
'waasamoo-bimide') under one predicate.
[Line 262 Sent 31 Node 3]: [L3 Syntax punct-causes-nonproj] Punctuation must not cause non-projectivity of nodes 
[Node<31#2, waaboozoog>]
[Line 282 Sent 33 Node 4]: [L3 Syntax punct-causes-nonproj] Punctuation must not cause non-projectivity of nodes 
[Node<33#2, ini>]
[Line 318 Sent 38 Node 3]: [L3 Syntax punct-causes-nonproj] Punctuation must not cause non-projectivity of nodes 
[Node<38#2, naa>]
Syntax errors: 9
*** FAILED *** with 9 errors

### Summary

Above, we saw how to go from a list of Ojibwe sentences and turn it into a fully usable `.conllu` format treebank. Some simple functionality such as visualization is already implemented, and this notebook will be updated when more concrete development on the `corpus.py` module has been done.